# Sine Wave - Sequence Forecasting Comparison

Notebook này chạy 4 kiến trúc thuộc họ RNN trên một dataset chuỗi để so sánh.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
tf.random.set_seed(42)
np.random.seed(42)


In [ ]:
df = pd.read_csv(DATA_DIR / "sine_wave.csv")
series = df["signal"].to_numpy().astype("float32")
window = 40
X, y = [], []
for i in range(len(series) - window):
    X.append(series[i:i+window])
    y.append(series[i+window])
X = np.array(X)[..., None]
y = np.array(y)
split = int(0.8 * len(X))
x_train, x_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
input_shape = x_train.shape[1:]


In [ ]:
def build_simple_rnn(input_shape):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.SimpleRNN(64),
        layers.Dense(1)
    ])

def build_birnn(input_shape):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Bidirectional(layers.SimpleRNN(64)),
        layers.Dense(1)
    ])

def build_gru(input_shape):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(64),
        layers.Dense(1)
    ])

def build_deep_rnn(input_shape):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.SimpleRNN(64, return_sequences=True),
        layers.SimpleRNN(64),
        layers.Dense(1)
    ])

builders = {
    "SimpleRNN": build_simple_rnn,
    "BiRNN": build_birnn,
    "GRU": build_gru,
    "DeepRNN": build_deep_rnn,
}

results = []
for name, builder in builders.items():
    model = builder(input_shape)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=5, batch_size=64, verbose=0)
    loss, mae = model.evaluate(x_test, y_test, verbose=0)
    results.append({"model": name, "test_mse": loss, "test_mae": mae})

results_df = pd.DataFrame(results).sort_values("test_mse")
results_df

sns.barplot(data=results_df, x="test_mse", y="model", palette="crest")
plt.title("Sequence Forecasting Comparison")
plt.show()
